# Notebook to Calculate Binding, Stability and Aggregation Features

## Import modules

In [1]:
import os; cwd = os.getcwd()
if cwd.split('/')[-1]=='pips-design-toolkit': os.chdir('./tools')
from utils.variables import data_folder, subfolders
from utils.utils import fetch_sequences_from_fasta, get_mutations
from feature_extraction.binding import get_yasara_binding_features
from feature_extraction.stability import  get_yasara_foldx_stability_features
from feature_extraction.aggregation import AggWaltz, AggTango

# Specify input files needed
Specify the names for:
* *Input directory* (`input_dir`): Input directory in which mutation list and structures are found
* *Output directory* (`output_dir`): Output directory to which calculations are saved
* *Input filename for mutations list* (`input_fname`):  '.txt' file found in input directory ('data/feature_extraction/Input/') directory, specifying list of mutations to apply, each separated by a newline (e.g. Q372A Q372C S373A), or list of positions to mutate (e.g. Q372 S373)
* *Sequence filename* (<`seq_fname`>.fasta): Fasta file found in 'data/sequences/' directory

In [2]:
# set inputs and parameters
input_dir = data_folder + 'feature_extraction/Input/'
output_dir = data_folder + 'feature_extraction/'
input_fname = 'GOh1052_mutPos_DomainIII_SNIPPET.txt'
sequence_fname = 'GOh1052'

# Get Binding features with YASARA

#### Prerequisites: 
* YASARA Structure installation (with necessary license)
* `yasara.py` file in './tools/', with yasara_dir location specified, e.g. yasaradir = '/Applications/YASARA.app/Contents/yasara/'

#### Additional inputs: 
* *Structure filename* (<`seq_fname`>.sce): YASARA scene file containing docked structure of receptor (as Object 1) with ligand (as Object 2). This should be placed in `input_dir`.
* *Structure directory* (`struct_dir`): Directory in which .sce structure input is placed. Default: './data/sce/'
* *Output filename* (<`output_fname`>.csv): Filename for output CSV, saved to `output_dir`.
* *List of backbone mutations to apply to input structure* (`backbone_mutations`): These backbone mutations would be applied on top of those specified in <`input_fname`>.txt. E.g. ['F194A','N245W']
* *Number of repetitions* (`nrep`): # of repetitions over which energy calculations are averaged in YASARA simulation

Results are saved at: <`output_dir`>/<`output_fname`>.csv

<img src="./data/img/binding.png" width="400" height="250">


In [10]:
# set inputs and parameters
struct_fname = 'S152_1GOG_GOh1001b_postOpt'
struct_dir = data_folder + subfolders['sce']
output_fname = 'DDGbind_GOh1052'
backbone_mutations = ['F194A','N245W'] # backbone mutations on top of GOh1001b structure to get GOh1052 (Offset by -24 wrt F217A+N268W)
nrep=5

# get binding ddG and energy features
get_yasara_binding_features(
    input_fname,
    struct_fname,
    output_fname,
    input_dir,
    struct_dir,
    output_dir,
    nrep,
    backbone_mutations
)

# of positions to mutate: 2 ['Q372', 'S373']
mutations by position: {'Q372': ['Q372A', 'Q372C'], 'S373': ['S373A']}
Starting Yasara processing for S152_1GOG_GOh1001b_postOpt >> F194A+N245W+Q372A
Set info mode
Set license shown
Set system time
Set info pid
Set Processors
Set Energy Unit
Set Force Field
rep#=0
Clear previous contents
Load Sce
Clean All
Actual WT residue: PHE ; Target WT residue: PHE
Correct WT amino acid found at position to mutate >> Continue processing
Actual WT residue: ASN ; Target WT residue: ASN
Correct WT amino acid found at position to mutate >> Continue processing
Actual WT residue: GLN ; Target WT residue: GLN
Correct WT amino acid found at position to mutate >> Continue processing
mutname: Phe194AlaAsn245TrpGln372Ala
Performing SwapRes for WT...
F194A
N245W
Q372A
Performing Energy Minimization for WT...
ExpEnd
Calculating energies for WT...
Processing element 3: Cpx
Processing element 1: Rtr
Processing element 2: Lgd
Obtained energies for WT
ebindingWT: [-279.

KeyboardInterrupt: 

# Get Stability features with YASARA + FoldX

#### Prerequisites: 
* YASARA Structure installation (see cell above)
* FoldX installation (with necessary license). Installation instructions here: https://foldxyasara.switchlab.org/index.php/FoldX_plugin_for_YASARA
* `yasara.py` file in './tools/', with FoldX executable location specified, e.g. foldx_abspath = '/Applications/YASARA.app/Contents/yasara/foldx_2025/foldx_20251231_mac'

#### Additional inputs: 
* *PDB filename* (<`input_pdb_fname`>.pdb): PDB file containing docked structure of receptor for assessing the effect of mutational changes on stability. This should be placed in `input_dir`.
* *Structure directory* (`struct_dir`): Directory in which .pdb structure input is placed. Default: './data/pdb/'
* *Output filename* (<`output_fname`>.csv): Filename for output CSV, saved to `output_dir`.

Results are saved at: <`output_dir`>/<`output_fname`>.csv

Note: If YASARA was previously run to calculate the Binding energy or Stability, then the notebook kernel needs to be restarted before running YASARA again. 

<img src="./data/img/stability.png" width="500" height="300">


In [ ]:
# define directories & filenames
input_pdb_fname = 'YASARA_2EIE_GOh1052.pdb'
struct_dir = data_folder + subfolders['pdb']
output_fname = 'DDGstability_GOh1052'

# get stability ddG and energy features
get_yasara_foldx_stability_features(
        input_fname,
        input_pdb_fname,
        output_fname,
        input_dir,
        struct_dir,
        output_dir,
        remove_existing_dir=False,
        receptor_molname='A',
)

# of positions to mutate: 2 ['Q372', 'S373']
mutations by position: {'Q372': ['Q372A', 'Q372C'], 'S373': ['S373A']}
Save folders: ../data/feature_extraction/SwapRes/Q372A/ ../data/feature_extraction/Repair/Q372A/ ../data/feature_extraction/Build/Q372A/
Started YASARA
Loaded PDB
Performed SwapRes  372 Gln for WT structure


# Get Aggregation features

<img src="./data/img/aggregation.png" width="600" height="220">

### Get base sequence and all possible single-site mutations

In [ ]:
# get base sequence(s)
seqs, seq_names, seq_description = fetch_sequences_from_fasta(f'{data_folder}sequences/{sequence_fname}.fasta')
seq_base = seqs[0]
seq_name = seq_names[0]

# get mutations
mutatePos = [aa+str(i+1) for i,aa in enumerate(seq_base)]
mutations = [None] + get_mutations(mutatePos)

### Amyloid aggregation prediction with Waltz
Prerequisites: 
* `Waltz.pl` perl script for running Waltz found in './tools/feature_extraction/aggregation/' folder

Note Perl script is run via python subprocess. If running on a non-Unix operating system (e.g. Windows), a Perl interpreter needs to first be installed for correct execution. 

Final results saved as CSV file: <`output_dir`>/AggregationScore_<`sequence_fname`>_waltz.csv

In [ ]:
AggWaltz(mutations, seq_base, seq_name, input_dir+'waltz/', output_dir, f'AggregationScore_{sequence_fname}')    

### Amorphous aggregation prediction with Tango
Prerequisites: 
* `Tango.exe` executable for running Tango should be placed in the './tools/feature_extraction/aggregation/' folder

Tango.exe is not provided in this repo and needs to be separately installed. The software license can be obtained here: https://tango.crg.es/

Final results saved as CSV file in <`output_dir`>/AggregationScore_<`sequence_fname`>_tango.csv'

In [ ]:
AggTango(mutations, seq_base, seq_name, input_dir+'tango/', output_dir, f'AggregationScore_{sequence_fname}')